In [ ]:

!pip install -q tensorflow tensorflow-hub easyocr opencv-python pillow transformers torch torchvision gradio fpdf pytesseract
!sudo apt-get -qq update
!sudo apt-get -qq install -y tesseract-ocr

print("✅ Install step finished.")


  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'fpdf' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'fpdf'. Discussion can be found at https://github.com/pypa/pip/issues/6334
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Install step finished.


In [ ]:
# Imports and global settings
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
import easyocr
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, MarianTokenizer, MarianMTModel
import gradio as gr
from fpdf import FPDF
import tempfile
import re
import os
from datetime import datetime
import pytesseract

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [ ]:
# Settings
IMG_SIZE = 224  # MobileNetV2 standard input size

# Load MobileNetV2 pretrained on ImageNet
model = tf.keras.applications.MobileNetV2(weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
model.trainable = False
print("✅ MobileNetV2 loaded.")

# Load ImageNet class names file
!wget -q -O imagenet_classes.txt https://storage.googleapis.com/download.tensorflow.org/data/ImageNetLabels.txt
with open('imagenet_classes.txt', 'r') as f:
    class_names = [line.strip() for line in f.readlines()]
print(f"✅ Loaded {len(class_names)} ImageNet class names.")

# Load BLIP captioner (image-to-text)
print("⏳ Loading BLIP captioner (this may take a bit)...")
captioner = pipeline("image-to-text", model="Salesforce/blip-image-captioning-base")
print("✅ BLIP captioner ready.")


14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
✅ MobileNetV2 loaded.
✅ Loaded 1001 ImageNet class names.
⏳ Loading BLIP captioner (this may take a bit)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2242: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


✅ BLIP captioner ready.


In [ ]:
# EasyOCR reader (Arabic + English)
reader = easyocr.Reader(['en', 'ar'], verbose=False)
print("✅ EasyOCR ready.")

# Text generator (for marketing descriptions) - using BART (seq2seq summarization model)
print("⏳ Loading BART summarization/generation model (may take time)...")
tokenizer_bart = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
print("✅ BART loaded.")

# Translation model English -> Arabic (Helsinki)
print("⏳ Loading translation model (en->ar)...")
trans_model_name = "Helsinki-NLP/opus-mt-en-ar"
trans_tokenizer = MarianTokenizer.from_pretrained(trans_model_name)
trans_model = MarianMTModel.from_pretrained(trans_model_name)
print("✅ Translation model loaded.")


✅ EasyOCR ready.
⏳ Loading BART summarization/generation model (may take time)...


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

✅ Translation model loaded.


In [ ]:
# Helper: load image from URL or local file and return PIL.Image and numpy array
def load_image_from_source(source, target_size=(IMG_SIZE, IMG_SIZE)):
    if isinstance(source, str) and source.lower().startswith('http'):
        headers = {'User-Agent': 'Mozilla/5.0'}
        resp = requests.get(source, headers=headers)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert('RGB')
    else:
        img = Image.open(source).convert('RGB')
    img_resized = img.resize(target_size)
    return img_resized, img  # resized for model, original for display

# Prepare image for MobileNetV2
def prepare_for_mobilenet(pil_img):
    arr = np.array(pil_img).astype(np.float32)
    arr = tf.keras.applications.mobilenet_v2.preprocess_input(arr)
    arr = np.expand_dims(arr, axis=0)
    return arr

# Predict with MobileNetV2 and return top-k
def classify_image(pil_img, top_k=5):
    tensor = prepare_for_mobilenet(pil_img)
    preds = model.predict(tensor)
    probs = tf.nn.softmax(preds[0]).numpy()
    top_indices = np.argsort(probs)[-top_k:][::-1]
    results = [(class_names[idx], float(probs[idx])) for idx in top_indices]
    top_class = results[0][0] if results else "Unknown"
    top_conf = results[0][1] if results else 0.0
    return top_class, top_conf, results

# Caption with BLIP
def caption_image(pil_img):
    try:
        out = captioner(pil_img)
        if out and isinstance(out, list) and 'generated_text' in out[0]:
            return out[0]['generated_text']
        # fallback: pipeline sometimes returns different key depending on version
        if out and isinstance(out, list) and len(out)>0:
            # try common keys
            for k in ['caption','generated_text','text','label']:
                if k in out[0]:
                    return out[0][k]
        return "No description generated."
    except Exception as e:
        return f"Caption Error: {e}"

# OCR extraction
def ocr_extract(pil_img):
    try:
        res = reader.readtext(np.array(pil_img))
        text = " ".join([r[1] for r in res]) if res else ""
        return text if text.strip() else "No text detected."
    except Exception as e:
        return f"OCR Error: {e}"

# Generate marketing text using BART (we use seq2seq generate)
def generate_marketing_text(cnn_result, caption, extracted_text):
    prompt = f"Product Category: {cnn_result}\nDescription: {caption}\nText on Product: {extracted_text}\n\nWrite a short, catchy marketing description for this product suitable for an online store."
    inputs = tokenizer_bart([prompt], max_length=512, return_tensors="pt", truncation=True)
    summary_ids = bart_model.generate(inputs["input_ids"], max_length=120, num_beams=4, early_stopping=True)
    output = tokenizer_bart.decode(summary_ids[0], skip_special_tokens=True)
    return output

# Translate to Arabic
def translate_to_arabic(text):
    if not text or text.strip() == "":
        return "لا يوجد نص للترجمة."
    inputs = trans_tokenizer([text], return_tensors="pt", padding=True, truncation=True)
    translated_tokens = trans_model.generate(**inputs, max_length=256)
    translated_text = trans_tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return translated_text

# Simple QA answer generator (very lightweight rule-based)
def simple_qa_answer(category, caption, ocr_text, question):
    q = question.strip().lower()
    if not q:
        return "Ask a question about the image (e.g., 'What is this?', 'What color?', 'Is there text?')."
    if "what" in q or "إيه" in q or "what is" in q:
        return f"This seems to be: {category}. ({caption})"
    if "color" in q or "لون" in q:
        # attempt get color words from caption
        return f"Description suggests: {caption}."
    if "text" in q or "نص" in q:
        return f"Detected text: {ocr_text}"
    if "size" in q or "حجم" in q:
        return f"Image was analyzed at {IMG_SIZE}x{IMG_SIZE}px; perceived size: {caption}."
    return f"Short answer: It looks like {category}. Caption: {caption}"


In [ ]:
def analyze_image(image_np):
    """
    image_np: numpy array image from Gradio input
    returns: cnn_result_str, caption_str, ocr_str, qa_default_answer, top_predictions (list)
    """
    if image_np is None:
        return "Please upload an image.", "", "", "", []
    pil_image = Image.fromarray(image_np).convert("RGB")
    pil_for_model = pil_image.resize((IMG_SIZE, IMG_SIZE))

    # CNN
    try:
        top_class, top_conf, top_list = classify_image(pil_for_model, top_k=5)
        cnn_result = f"{top_class} ({top_conf*100:.2f}%)"
    except Exception as e:
        top_list = []
        cnn_result = f"CNN Error: {e}"
        top_class = "Unknown"

    # Caption
    caption = caption_image(pil_image)

    # OCR
    ocr_text = ocr_extract(pil_image)

    # Simple QA default
    qa_default = simple_qa_answer(top_class, caption, ocr_text, "What is this?")

    # Format top predictions nicely
    top_preds_text = "\n".join([f"{i+1}. {name} ({score*100:.2f}%)" for i,(name,score) in enumerate(top_list)])

    return cnn_result, caption, ocr_text, qa_default, top_preds_text


In [ ]:
# Utilities to save files and reports
def safe_filename_from_cnn(cnn_result):
    if not cnn_result:
        return "Product"
    # remove parentheses/confidence, keep words
    name = str(cnn_result).split("(")[0].strip()
    clean_name = re.sub(r'[^a-zA-Z0-9_\u0600-\u06FF ]+', '', name).strip().replace(" ", "_")
    return clean_name if clean_name else "Product"

def save_text_file(text, filename):
    path = os.path.join(tempfile.gettempdir(), filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return path

def save_pdf_text(text, filename):
    path = os.path.join(tempfile.gettempdir(), filename)
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.set_font("Arial", size=12)
    for line in text.split("\n"):
        pdf.multi_cell(0, 8, line)
    pdf.output(path)
    return path

def save_full_report_with_cover(image_np, cnn_result, caption, ocr_text, qa_text, marketing_en, marketing_ar):
    if image_np is None:
        return None, "Please provide an image first."

    clean_name = safe_filename_from_cnn(cnn_result)
    filename = f"{clean_name}_Smart_Analysis_Report.pdf"
    temp_path = os.path.join(tempfile.gettempdir(), filename)

    # save image temporarily
    pil_image = Image.fromarray(image_np.astype(np.uint8))
    temp_image_path = os.path.join(tempfile.gettempdir(), f"{clean_name}_image.jpg")
    pil_image.save(temp_image_path)

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)

    # Cover page
    pdf.add_page()
    pdf.set_font("Arial", "B", 22)
    pdf.cell(0, 20, "🧠 Smart Product Analyzer", ln=True, align="C")
    pdf.ln(10)
    pdf.set_font("Arial", "", 16)
    pdf.cell(0, 10, f"📦 Product: {cnn_result}", ln=True, align="C")
    pdf.cell(0, 10, f"👤 Report by: Kahkaya", ln=True, align="C")
    pdf.cell(0, 10, f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", ln=True, align="C")
    pdf.ln(8)
    # add image
    pdf.image(temp_image_path, x=35, y=80, w=140)
    pdf.ln(95)
    pdf.set_font("Arial", "I", 12)
    pdf.cell(0, 10, "AI-powered product analysis report.", ln=True, align="C")

    # Details page
    pdf.add_page()
    pdf.set_font("Arial", "B", 16)
    pdf.cell(0, 10, "🔍 Detailed Analysis", ln=True, align="C")
    pdf.ln(8)

    def add_section(title, content):
        pdf.set_font("Arial", "B", 14)
        pdf.cell(0, 10, title, ln=True)
        pdf.set_font("Arial", "", 12)
        pdf.multi_cell(0, 8, content if content else "No data available.")
        pdf.ln(5)

    add_section("1️⃣ CNN Classification Result:", cnn_result)
    add_section("2️⃣ Image Description (BLIP):", caption)
    add_section("3️⃣ Detected Text (OCR):", ocr_text)
    add_section("4️⃣ Simple Q&A Output:", qa_text)
    add_section("5️⃣ Marketing Description (English):", marketing_en)
    add_section("6️⃣ Marketing Description (Arabic):", marketing_ar)

    pdf.output(temp_path)
    return temp_path, f"Full report saved: {filename}"

# Save marketing text only (txt or pdf) with product-based name
def save_marketing_file(marketing_en, marketing_ar, language, file_type, cnn_result):
    clean_name = safe_filename_from_cnn(cnn_result)
    if language == "English":
        text_to_save = marketing_en
        filename = f"{clean_name}_Description.{file_type}"
    elif language == "Arabic":
        text_to_save = marketing_ar
        filename = f"{clean_name}_Description_AR.{file_type}"
    else:
        text_to_save = f"English:\n{marketing_en}\n\nArabic:\n{marketing_ar}"
        filename = f"{clean_name}_Description_BOTH.{file_type}"

    if not text_to_save or text_to_save.strip() == "":
        return None, "No marketing text to save."

    path = os.path.join(tempfile.gettempdir(), filename)
    if file_type == "txt":
        with open(path, "w", encoding="utf-8") as f:
            f.write(text_to_save)
    else:
        # pdf
        pdf = FPDF()
        pdf.add_page()
        pdf.set_auto_page_break(auto=True, margin=15)
        pdf.set_font("Arial", size=12)
        for line in text_to_save.split("\n"):
            pdf.multi_cell(0, 8, line)
        pdf.output(path)
    return path, f"Saved file: {filename}"


In [ ]:
# Gradio UI wiring
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Smart Product Analyzer (By Kahkaya)")
    gr.Markdown("Upload an image — the system will classify it, describe it, extract text, generate marketing copy, translate, and let you save a full report.")

    with gr.Row():
        image_input = gr.Image(type="numpy", label="Upload Image")
        analyze_btn = gr.Button("🔍 Analyze", variant="primary")

    with gr.Row():
        cnn_output = gr.Textbox(label="🧠 CNN Classification (top result)")
        caption_output = gr.Textbox(label="📝 BLIP Description")
        ocr_output = gr.Textbox(label="🔤 OCR Text")
        qa_output = gr.Textbox(label="💬 Simple Answer (default)")

    with gr.Row():
        top_preds_output = gr.Textbox(label="📊 Top Predictions (top 5)", lines=5)

    with gr.Row():
        language_choice = gr.Radio(choices=["English", "Arabic", "Both"], value="Both", label="🌍 Marketing Text Language")
        file_type_choice = gr.Radio(choices=["txt", "pdf"], value="txt", label="📄 Save as")

    with gr.Row():
        marketing_btn = gr.Button("✨ Generate Marketing Text", variant="secondary")

    with gr.Row():
        marketing_output_en = gr.Textbox(label="📣 Marketing Description (English)", lines=4)
        marketing_output_ar = gr.Textbox(label="🗣️ Marketing Description (Arabic)", lines=4)

    with gr.Row():
        save_marketing_btn = gr.Button("💾 Save Marketing Text to File")
        download_marketing_file = gr.File(label="⬇️ Download Marketing File")
        save_marketing_msg = gr.Textbox(label="📢 Status", interactive=False)

    with gr.Row():
        save_report_btn = gr.Button("📘 Save Full Report (By Kahkaya)")
        download_report_file = gr.File(label="⬇️ Download Full Report (PDF)")
        save_report_msg = gr.Textbox(label="📢 Report Status", interactive=False)

    # Hook functions
    def on_analyze(image_np):
        return analyze_image(image_np)

    def on_generate_marketing(cnn_result, caption, ocr_text, top_preds):
        # cnn_result is like "tabby cat (92.00%)" -> extract name
        top_class = cnn_result.split("(")[0].strip() if cnn_result else "Product"
        marketing_en = generate_marketing_text(top_class, caption, ocr_text)
        marketing_ar = translate_to_arabic(marketing_en)
        return marketing_en, marketing_ar

    def on_save_marketing(marketing_en, marketing_ar, language, file_type, cnn_result):
        path, msg = save_marketing_file(marketing_en, marketing_ar, language, file_type, cnn_result)
        if path:
            return path, msg
        return None, msg

    def on_save_report(image_np, cnn_result, caption, ocr_text, qa_text, marketing_en, marketing_ar):
        path, msg = save_full_report_with_cover(image_np, cnn_result, caption, ocr_text, qa_text, marketing_en, marketing_ar)
        if path:
            return path, msg
        return None, msg

    analyze_btn.click(fn=on_analyze, inputs=[image_input], outputs=[cnn_output, caption_output, ocr_output, qa_output, top_preds_output])
    marketing_btn.click(fn=on_generate_marketing, inputs=[cnn_output, caption_output, ocr_output, top_preds_output], outputs=[marketing_output_en, marketing_output_ar])
    save_marketing_btn.click(fn=on_save_marketing, inputs=[marketing_output_en, marketing_output_ar, language_choice, file_type_choice, cnn_output], outputs=[download_marketing_file, save_marketing_msg])
    save_report_btn.click(fn=on_save_report, inputs=[image_input, cnn_output, caption_output, ocr_output, qa_output, marketing_output_en, marketing_output_ar], outputs=[download_report_file, save_report_msg])

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4d83cc5083959d98e3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 745, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 56, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 745, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 56, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 745, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 56, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^